# 02 - Decision-Driven EDA

This notebook is focused on achieving the following:
- defining missing/unknown strategy
- validating encoding direction
- producing measurable evaluation plan (AUC, F1, confusion matrix, CV)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)

BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = BASE_DIR / 'data' / 'bank-full.csv'
FEATURES = ['age', 'job', 'default', 'housing', 'loan', 'marital', 'education']
TARGET = 'y'
CAT_FEATURES = ['job', 'default', 'housing', 'loan', 'marital', 'education']

In [ ]:
df = pd.read_csv(DATA_PATH, sep=';')
df = df[FEATURES + [TARGET]].copy()

print('Shape:', df.shape)
display(df.head())

## 1) Target Health

In [ ]:
target_dist = df[TARGET].value_counts(dropna=False).rename_axis('class').to_frame('count')
target_dist['pct'] = target_dist['count'] / target_dist['count'].sum()
display(target_dist)

baseline_majority = target_dist['pct'].max()
print('Majority-class baseline accuracy:', round(baseline_majority, 4))

ax = sns.countplot(data=df, x=TARGET)
ax.set_title('Target Distribution')
plt.show()

## 2) Data Quality and Unknown Audit

In [ ]:
quality = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'null_count': df.isna().sum(),
    'null_pct': df.isna().mean()
})
display(quality)

unknown_rows = []
for col in CAT_FEATURES:
    count_unknown = (df[col] == 'unknown').sum()
    unknown_rows.append({'feature': col, 'unknown_count': count_unknown, 'unknown_pct': count_unknown / len(df)})

unknown_df = pd.DataFrame(unknown_rows).sort_values('unknown_pct', ascending=False)
display(unknown_df)

duplicate_count = df.duplicated().sum()
print('Duplicate rows:', duplicate_count)

## 3) Category Frequency and Response Rate

In [ ]:
def category_profile(data, feature, target='y', positive_label='yes'):
    g = data.groupby(feature, dropna=False)[target].agg(['count'])
    g['pct'] = g['count'] / len(data)
    g['response_rate'] = data.groupby(feature, dropna=False)[target].apply(lambda s: (s == positive_label).mean())
    return g.sort_values('count', ascending=False)

for col in CAT_FEATURES:
    print(f'\n=== {col} ===')
    display(category_profile(df, col, target=TARGET))

## 4) Age Signal Check

In [ ]:
display(df['age'].describe())

age_bins = pd.cut(df['age'], bins=[17, 24, 34, 44, 54, 64, 100], include_lowest=True)
age_profile = df.assign(age_bin=age_bins).groupby('age_bin')[TARGET].agg(count='count', response_rate=lambda s: (s == 'yes').mean())
display(age_profile)

ax = sns.histplot(data=df, x='age', hue=TARGET, bins=25, stat='density', common_norm=False)
ax.set_title('Age Distribution by Target')
plt.show()

## 5) Split Diagnostics (Unseen Category Risk)

In [ ]:
X = df[FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Target rate (train):', round((y_train == 'yes').mean(), 4))
print('Target rate (test) :', round((y_test == 'yes').mean(), 4))

coverage_rows = []
for col in CAT_FEATURES:
    train_cats = set(X_train[col].unique())
    test_cats = set(X_test[col].unique())
    unseen_in_test = sorted(list(test_cats - train_cats))
    coverage_rows.append({
        'feature': col,
        'train_unique': len(train_cats),
        'test_unique': len(test_cats),
        'unseen_in_test_count': len(unseen_in_test),
        'unseen_in_test_values': ', '.join(unseen_in_test) if unseen_in_test else ''
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

## 6) Decision Table (Output for `train.py` Changes)

In [ ]:
decision_table = pd.DataFrame({
    'feature': FEATURES,
    'missing_strategy': ['none'] + ['keep_as_category_or_impute'] * 6,
    'unknown_strategy': ['n/a'] + ['to_confirm_from_unknown_rate'] * 6,
    'encoding': ['numeric_passthrough'] + ['one_hot'] * 6,
    'notes': ['confirm age scaling necessity'] + ['confirm rare-category threshold'] * 6
})
display(decision_table)

out_dir = BASE_DIR / 'notebooks' / 'artifacts'
out_dir.mkdir(parents=True, exist_ok=True)
decision_table.to_csv(out_dir / '02_eda_decisions.csv', index=False)
unknown_df.to_csv(out_dir / '02_eda_unknown_rates.csv', index=False)
coverage_df.to_csv(out_dir / '02_eda_split_coverage.csv', index=False)
print('Saved artifacts to', out_dir)

## 7) Stage 1 Implementation Notes

After running this notebook, update `train.py` with:
1. Chosen missing/unknown handling policy
2. Improved categorical encoding path
3. Metrics: Accuracy, F1, confusion matrix, AUC-ROC
4. Stratified cross-validation with a small hyperparameter grid